# Core Engine Module - Dependency Injection and Startup

This notebook demonstrates how to use the `core.engine` module, which provides:
- **InjectionContainer**: Dependency injection system
- **Startup**: Application initialization and database setup
- **Decorators**: Utility decorators for functions and classes

## Table of Contents
1. [Dependency Injection](#dependency-injection)
2. [Database Registration](#database-registration)
3. [Application Startup](#application-startup)
4. [Scoped Containers](#scoped-containers)

In [1]:

# Import required modules
import os, sys 
sys.path.append(os.path.abspath(".."))

from core.engine.context import (
    context,
    register,
    get,
    register_factory,
    container_scope,
    register_postgres,
    get_database_manager
)
from core.engine.startup import Startup
from core.engine.decorators import singleton, inject
from loguru import logger

print("✅ Imports successful!")

2026-01-18 07:44:47.703 | INFO     | core.database.registry:register:63 - Registered database backend: postgres
2026-01-18 07:44:47.705 | INFO     | core.database.registry:register:63 - Registered database backend: milvus
2026-01-18 07:44:47.705 | INFO     | core.database.registry:register:63 - Registered database backend: qdrant
2026-01-18 07:44:47.706 | INFO     | core.database.registry:register:63 - Registered database backend: mongo
2026-01-18 07:44:47.706 | INFO     | core.database.registry:register:63 - Registered database backend: milvus_async
2026-01-18 07:44:47.707 | INFO     | core.database.registry:register:63 - Registered database backend: qdrant_async
2026-01-18 07:44:47.708 | INFO     | core.database.registry:register:63 - Registered database backend: mongo_async
2026-01-18 07:44:47.709 | INFO     | core.database.plugins:register_builtin_backends:121 - Registered built-in database backends
2026-01-18 07:44:47.711 | WARNING  | core.database.plugins:discover_database_plugin

✅ Imports successful!


## 1. Dependency Injection

The `InjectionContainer` provides a simple dependency injection system.

In [2]:
# Example 1: Register and retrieve dependencies

# Register a simple dependency
register("app_name", "MyApplication")
register("version", "1.0.0")

# Retrieve dependencies
app_name = get("app_name")
version = get("version")

print(f"Application: {app_name}, Version: {version}")

Application: MyApplication, Version: 1.0.0


In [ ]:
# Example 2: Register a factory function
def create_logger(name: str):
    """Factory function to create a logger"""
    return logger.bind(name=name)

register_factory("logger", lambda: create_logger("my_app"))

# Get logger (created on demand)
app_logger = get("logger")
print(f"Logger created: {app_logger}")

## 2. Database Registration

Register and manage database connections through the DI container.

In [3]:
# Example: Register a PostgreSQL database
# Note: This requires actual database credentials

# Option 1: Register with explicit parameters
# register_postgres(
#     "main_db",
#     host="localhost",
#     port="5432",
#     database="mydb",
#     user="postgres",
#     password="password"
# )

# Option 2: Register using auto-config (reads from environment/config)
# register_postgres("main_db")  # Uses POSTGRES_HOST, POSTGRES_PORT, etc. from config

# Get database manager
# db_manager = get_database_manager("main_db")
# print(f"Database manager: {db_manager}")

print("💡 Database registration examples shown (commented out)")

💡 Database registration examples shown (commented out)


## 3. Application Startup

The `Startup` class provides utilities for initializing your application.

In [4]:
# Example: Initialize application with database setup
# This would:
# 1. Discover entities from 'entities' package
# 2. Register databases from config
# 3. Create database tables

# Startup.initialize(
#     create_tables=True,
#     discover_entities=True,
#     entities_package='entities'
# )

print("💡 Startup initialization example shown (commented out)")

💡 Startup initialization example shown (commented out)


## 4. Scoped Containers

Use scoped containers for isolated dependency injection contexts (useful for testing).

In [5]:
# Example: Using scoped containers

# Register in default context
register("default_dep", "default_value")

# Use scoped container
with container_scope("test"):
    # Register in test scope
    register("test_dep", "test_value")
    
    # Access test-scoped dependency
    test_value = get("test_dep")
    print(f"In test scope: {test_value}")
    
    # Default dep not available in test scope
    try:
        default_value = get("default_dep")
        print(f"Default value: {default_value}")
    except KeyError:
        print("Default dep not available in test scope (expected)")

# Back to default context
default_value = get("default_dep")
print(f"Back in default scope: {default_value}")

2026-01-18 07:46:51.825 | INFO     | core.database.registry:register:63 - Registered database backend: postgres
2026-01-18 07:46:51.827 | INFO     | core.database.registry:register:63 - Registered database backend: milvus
2026-01-18 07:46:51.827 | INFO     | core.database.registry:register:63 - Registered database backend: qdrant
2026-01-18 07:46:51.828 | INFO     | core.database.registry:register:63 - Registered database backend: mongo
2026-01-18 07:46:51.828 | INFO     | core.database.registry:register:63 - Registered database backend: milvus_async
2026-01-18 07:46:51.828 | INFO     | core.database.registry:register:63 - Registered database backend: qdrant_async
2026-01-18 07:46:51.829 | INFO     | core.database.registry:register:63 - Registered database backend: mongo_async
2026-01-18 07:46:51.829 | INFO     | core.database.plugins:register_builtin_backends:121 - Registered built-in database backends
2026-01-18 07:46:51.829 | WARNING  | core.database.plugins:discover_database_plugin

In test scope: test_value
Default dep not available in test scope (expected)
Back in default scope: default_value


In [6]:
# Example: Singleton decorator
@singleton
class DatabaseConnection:
    """Singleton database connection"""
    def __init__(self):
        self.connected = False
    
    def connect(self):
        self.connected = True
        print("Connected to database")

# Create instances - should return same instance
db1 = DatabaseConnection()
db2 = DatabaseConnection()

print(f"Same instance: {db1 is db2}")
db1.connect()
print(f"db2 also connected: {db2.connected}")

Same instance: True
Connected to database
db2 also connected: True


## Summary

**Dependency Injection**:
- Use `register()` to register dependencies
- Use `get()` to retrieve dependencies
- Use `@inject()` decorator for automatic injection

**Database Management**:
- Use `register_postgres()`, `register_mongo()`, etc. to register databases
- Use `get_database_manager()` to get database managers

**Scoped Containers**:
- Use `container_scope()` for isolated contexts
- Useful for testing and multi-tenant applications

